# Jueces open-weight: Llama 3.1 8B y Mistral 7B sobre las 18.000 citas

Este cuaderno agrega **dos jueces independientes** al pre-etiquetado de funciones de cita, como pide la actividad A2 del enunciado (sistema de jueces: varios modelos clasifican el mismo ejemplo y se contrastan sus respuestas).

- **Juez 1:** `meta-llama/Llama-3.1-8B-Instruct`
- **Juez 2:** `mistralai/Mistral-7B-Instruct-v0.3`
- **Juez 3 (ya existe):** Qwen2.5-7B ajustado con QLoRA, cuyas predicciones vienen en `ANOTACION_18000_CITAS_REALES_QLORA.csv`.

Cada juez responde con **una letra** (9 funciones + Ambiguo + No evaluable). No se genera texto: se leen las probabilidades de las 11 letras en una sola pasada del modelo, igual que en el cuaderno QLoRA. Así la salida siempre es válida y trae una confianza.

Todo se guarda en **Google Drive** mientras corre. Si Colab se desconecta, al volver a ejecutar la celda el juez sigue donde iba.

**Importante:** estas etiquetas siguen siendo pre-etiquetas de modelos. No reemplazan la anotación humana del Test Gold.

## Paso 0. Lo que necesitas antes de empezar

1. **Colab con GPU:** menú *Entorno de ejecución → Cambiar tipo de entorno*. Con Colab Pro, **A100**; en la T4 gratuita también corre, pero mucho más lento.
2. **Cuenta de Hugging Face** y aceptar la licencia de Llama, que es "gated" (Mistral v0.3 es Apache-2.0 y no pide aprobación):
   - https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct

   La aprobación de Llama puede tardar desde minutos hasta horas.
3. **Token de Hugging Face** de tipo *Read* (https://huggingface.co/settings/tokens). En Colab: icono de la llave (Secretos) → nuevo secreto llamado `HF_TOKEN` → pegar el token → activar el acceso del cuaderno. **No pegues el token en una celda.**
4. **El CSV en Drive:** sube `ANOTACION_18000_CITAS_REALES_QLORA.csv` a la carpeta `MyDrive/proyecto_grado/jueces/` (o cambia `CARPETA_DRIVE` abajo).

**Tiempo medido (3-nov-2026):** en una T4, unas 17 horas por juez (0,3 filas/s); en una A100, los dos jueces terminaron en menos de 2 horas. Antes de la corrida completa siempre se hace la prueba corta y el diagnóstico (Pasos 4 y 4b).

In [ ]:
# Comprobar que hay GPU asignada (debe aparecer una NVIDIA, por ejemplo Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Instalar las librerías que no trae Colab: bitsandbytes cuantiza el modelo a 4 bits para que quepa en la GPU
!pip install -q -U transformers accelerate bitsandbytes

## Paso 1. Configuración y datos

In [ ]:
import os, json, time, math, hashlib, random                         # utilidades estándar
from collections import Counter                                      # conteos de etiquetas
from pathlib import Path                                             # rutas multiplataforma

import pandas as pd                                                  # lectura y escritura del CSV
from google.colab import drive                                       # acceso a Google Drive desde Colab

drive.mount('/content/drive')                                        # monta Drive; pide autorización la primera vez

CARPETA_DRIVE = Path('/content/drive/MyDrive/proyecto_grado/jueces')  # carpeta donde están el CSV y donde se guardan los resultados
CSV_ENTRADA   = CARPETA_DRIVE / 'ANOTACION_18000_CITAS_REALES_QLORA.csv'  # las 18.000 citas con la predicción QLoRA
CARPETA_DRIVE.mkdir(parents=True, exist_ok=True)                     # crea la carpeta si no existe

JUECES = {                                                           # nombre corto -> modelo en Hugging Face
    'llama':   'meta-llama/Llama-3.1-8B-Instruct',
    'mistral': 'mistralai/Mistral-7B-Instruct-v0.3',
}

RESUMEN_CHARS = 400      # se recorta el resumen del artículo citado (mismo valor que en el cuaderno QLoRA)
LOTE          = 8       # filas por pasada; si aparece "CUDA out of memory", bájalo a 4
SEMILLA       = 42       # se fija para que la prueba corta tome siempre las mismas filas

assert CSV_ENTRADA.exists(), f'No encuentro {CSV_ENTRADA}: súbelo a Drive o corrige CARPETA_DRIVE'

In [ ]:
# Leer el CSV (utf-8-sig quita la marca BOM que deja Excel)
df = pd.read_csv(CSV_ENTRADA, encoding='utf-8-sig', dtype=str, keep_default_na=False)

COL = {'id': 'context_id', 'ctx': 'Contexto TARGETCIT', 'titulo': 'Título citado',
       'resumen': 'Resumen citado', 'ev': '¿Evaluable?', 'fn': 'Función principal', 'just': 'Justificación / duda'}
faltan = [c for c in COL.values() if c not in df.columns]           # comprobar que el archivo es el esperado
assert not faltan, f'Faltan columnas: {faltan}'
assert df[COL['id']].is_unique, 'Hay context_id repetidos'          # cada cita debe aparecer una sola vez

print(len(df), 'citas')                                              # deben ser 18.000
print('Predicción QLoRA (función):', df[COL['fn']].value_counts().to_dict())

## Paso 2. Taxonomía y prompt

Las definiciones resumen la tabla de funciones del enunciado del Tema 1. El prompt es **básico y sin ejemplos** (zero-shot), como pide A2: así los jueces no copian el criterio del modelo QLoRA ni de Haiku.

In [ ]:
OPCIONES = [   # (etiqueta, definición resumida de la tabla del enunciado)
    ('Background', 'provides context, summarizes the general background of a topic, or traces the history of a field or idea'),
    ('Gap', 'helps identify research gaps or unexplored areas, justifying the choice of research topic'),
    ('Basis', 'provides a foundation for the current research (broad influence or a specific key contribution) that the work builds upon'),
    ('Comparison', 'compares the current work with the cited study, or compares cited studies with each other'),
    ('Application', 'directly employs a method, technique, tool or data from the cited work without modification'),
    ('Improvement / Modification', 'adapts, expands, improves or modifies a method or tool from the cited work'),
    ('Evidence', 'supports claims, hypotheses, findings or methodological decisions of the current research'),
    ('Identification of the Originator', 'acknowledges the original source of an idea, concept, method or theory'),
    ('Further Reading', 'directs readers to additional or supplementary literature for more details'),
    ('Ambiguo', 'ambiguous: two or more readings are plausible, or the truncated context does not allow a decision'),
    ('No evaluable', 'not evaluable: the context gives no information about the function of this citation'),
]
ETIQUETAS = [e for e, _ in OPCIONES]                                 # 11 etiquetas en orden fijo
LETRAS = 'ABCDEFGHIJK'                                               # una letra por etiqueta
assert len(LETRAS) == len(ETIQUETAS)

def prompt_usuario(fila, opciones=OPCIONES):
    """Arma el mensaje del usuario para una cita (las opciones se pueden reordenar para medir el sesgo de posición)."""
    resumen = fila[COL['resumen']][:RESUMEN_CHARS]                  # el resumen se recorta para no alargar el prompt
    lineas = [
        'Classify the rhetorical function of the citation marked TARGETCIT in a computational linguistics paper.',
        'OTHERCIT marks other citations. The context may be truncated.',
        '',
        f'Cited paper title: {fila[COL["titulo"]]}',
        f'Cited paper abstract (truncated): {resumen}',
        f'Citation context: {fila[COL["ctx"]]}',
        '',
        'Options:',
    ]
    lineas += [f'{l}) {e} - {d}' for l, (e, d) in zip(LETRAS, opciones)]   # lista de opciones con su letra
    lineas += ['', 'Answer with a single letter from A to K.']
    return '\n'.join(lineas)

HASH_PROMPT = hashlib.sha256(prompt_usuario(df.iloc[0]).encode()).hexdigest()[:12]   # huella para documentar qué prompt se usó
print(prompt_usuario(df.iloc[0]))

## Paso 3. Funciones del juez (carga, predicción y guardado reanudable)

In [ ]:
import torch                                                         # PyTorch ya viene en Colab
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))                                # el token se lee del secreto, nunca se escribe en el cuaderno

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16   # la T4 no soporta bfloat16; usa float16

def cargar(modelo_id):
    """Carga el modelo cuantizado a 4 bits y su tokenizador."""
    tok = AutoTokenizer.from_pretrained(modelo_id)
    tok.padding_side = 'left'                                        # con relleno a la izquierda, la última posición es la respuesta
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token                                # Llama y Mistral no traen token de relleno
    cuant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                               bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    modelo = AutoModelForCausalLM.from_pretrained(modelo_id, quantization_config=cuant, device_map={'': 0}, dtype=DTYPE)
    modelo.eval()
    # Cada letra puede tokenizarse sola ("A") o con espacio delante (" A"); se suman ambas variantes
    ids_letra = []
    for l in LETRAS:
        variantes = {tok.encode(l, add_special_tokens=False)[-1], tok.encode(' ' + l, add_special_tokens=False)[-1]}
        ids_letra.append(sorted(variantes))
    return tok, modelo, ids_letra

def texto_chat(tok, fila, opciones=OPCIONES, prefijo=''):
    """Aplica la plantilla de chat del modelo (solo mensaje de usuario, que ambos modelos aceptan).
    `prefijo` es texto que se deja escrito al inicio de la respuesta del modelo (por ejemplo 'Answer:')."""
    mensajes = [{'role': 'user', 'content': prompt_usuario(fila, opciones)}]
    return tok.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True) + prefijo

@torch.no_grad()
def predecir(tok, modelo, ids_letra, filas):
    """Devuelve, para cada fila, la letra más probable y la distribución sobre las 11 opciones."""
    textos = [texto_chat(tok, f) for f in filas]
    enc = tok(textos, return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')   # sin recorte: la respuesta va al final
    try:
        logits = modelo(**enc, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()   # solo el último token: ahorra memoria
    except TypeError:
        logits = modelo(**enc, use_cache=False).logits[:, -1, :].float()                     # versiones antiguas de transformers
    puntajes = torch.stack([torch.logsumexp(logits[:, ids], dim=-1) for ids in ids_letra], dim=-1)
    probs = puntajes.softmax(-1).cpu()                                # probabilidad sobre las 11 letras
    salida = []
    for p in probs:
        k = int(p.argmax())
        salida.append({'etiqueta': ETIQUETAS[k], 'p': round(float(p[k]), 4), 'dist': [round(float(x), 4) for x in p]})
    return salida

def leer_hechos(ruta):
    """Lee las predicciones ya guardadas para poder reanudar."""
    hechos = {}
    if ruta.exists():
        for linea in ruta.read_text(encoding='utf-8').splitlines():
            if linea.strip():
                d = json.loads(linea)
                hechos[d['context_id']] = d
    return hechos

def correr_juez(nombre, max_filas=None):
    """Clasifica las citas pendientes con un juez y las va guardando en Drive (archivo JSONL)."""
    ruta = CARPETA_DRIVE / f'juez_{nombre}.jsonl'
    hechos = leer_hechos(ruta)
    pendientes = df[~df[COL['id']].isin(hechos)].copy()
    if max_filas:
        pendientes = pendientes.sample(n=min(max_filas, len(pendientes)), random_state=SEMILLA)   # prueba corta reproducible
    pendientes['_largo'] = pendientes[COL['ctx']].str.len() + pendientes[COL['resumen']].str.len().clip(upper=RESUMEN_CHARS)
    pendientes = pendientes.sort_values('_largo')                    # lotes de largo parecido = menos relleno y más velocidad
    print(f'{nombre}: {len(hechos)} ya hechas, {len(pendientes)} por procesar')
    if len(pendientes) == 0:
        return hechos
    tok, modelo, ids_letra = cargar(JUECES[nombre])
    filas = [f for _, f in pendientes.iterrows()]
    t0 = time.time()
    with open(ruta, 'a', encoding='utf-8') as salida:
        for i in range(0, len(filas), LOTE):
            lote = filas[i:i + LOTE]
            for f, r in zip(lote, predecir(tok, modelo, ids_letra, lote)):
                registro = {'context_id': f[COL['id']], 'juez': nombre, **r}
                salida.write(json.dumps(registro, ensure_ascii=False) + '\n')
                hechos[registro['context_id']] = registro
            salida.flush()                                           # se escribe en Drive en cada lote
            hechas = i + len(lote)
            if hechas % (LOTE * 50) < LOTE or hechas == len(filas):
                vel = hechas / (time.time() - t0)
                print(f'  {hechas}/{len(filas)}  {vel:.1f} filas/s  faltan ~{(len(filas) - hechas) / vel / 60:.0f} min')
    meta = {'juez': nombre, 'modelo': JUECES[nombre], 'hash_prompt': HASH_PROMPT, 'cuantizacion': 'nf4 4-bit',
            'dtype': str(DTYPE), 'filas': len(hechos), 'fecha': time.strftime('%Y-%m-%d %H:%M'),
            'transformers': __import__('transformers').__version__, 'torch': torch.__version__,
            'gpu': torch.cuda.get_device_name(0)}
    (CARPETA_DRIVE / f'juez_{nombre}_meta.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False))
    del modelo                                                       # libera la GPU para el siguiente juez
    torch.cuda.empty_cache()
    return hechos

## Paso 4. Prueba corta (64 filas por juez)

Revisa que la distribución no salga toda en una sola letra. Si un juez responde casi siempre lo mismo, avisa antes de correr las 18.000.

In [ ]:
for nombre in JUECES:
    prueba = correr_juez(nombre, max_filas=64)
    print(nombre, Counter(d['etiqueta'] for d in prueba.values()).most_common())

## Paso 4b. Diagnóstico: qué responde realmente cada juez

La corrida v1 (3-nov-2026) salió inservible: los jueces casi nunca eligieron la letra A (*Background*) ni la E (*Application*), y el acuerdo con el QLoRA quedó en el nivel del azar (κ de Fleiss −0,014). Esta celda averigua por qué, con 22 citas (2 por cada etiqueta del QLoRA), y compara cuatro cosas:

1. **v1:** el método de la corrida v1 (probabilidad de cada letra en la primera palabra de la respuesta). `masa_v1` dice cuánta probabilidad le da el modelo a las 11 letras juntas: si es baja, el modelo quería escribir otra cosa (por ejemplo "The answer is…") y la v1 estaba leyendo ruido.
2. **prefijo:** lo mismo, pero con `Answer:` ya escrito al inicio de la respuesta, para que lo siguiente sea la letra.
3. **invertido:** igual que *prefijo*, con las opciones en orden inverso (A = No evaluable … K = Background). Si la etiqueta elegida cambia al mover las opciones, el modelo elige por posición y no por contenido.
4. **texto_libre:** lo que el modelo escribe si se le deja generar 24 tokens.

También muestra qué tokens se usan para cada letra y compara la predicción una por una contra la predicción en lotes (para descartar un problema de relleno).

No escribe en los archivos de los jueces. Guarda `diagnostico_jueces.csv` en la carpeta de Drive.

In [ ]:
import gc                                                            # liberar memoria entre un juez y otro

q_todas = df[COL['fn']].where(df[COL['ev']] == 'Evaluable', df[COL['ev']])   # etiqueta del QLoRA en 11 clases
muestra = pd.concat([g.sample(min(len(g), 2), random_state=SEMILLA)          # 2 citas por etiqueta, siempre las mismas
                     for _, g in df.assign(_q=q_todas).groupby('_q')])
OPC_INV = OPCIONES[::-1]                                             # opciones en orden inverso
ETQ_INV = [e for e, _ in OPC_INV]                                    # etiqueta que corresponde a cada letra en ese orden
PREFIJO = 'Answer:'                                                  # texto que se deja escrito al inicio de la respuesta

@torch.no_grad()
def ultimo_logit(modelo, enc):
    """Logits de la última posición: ahí el modelo escribiría el siguiente token de su respuesta."""
    try:
        return modelo(**enc, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
    except TypeError:
        return modelo(**enc, use_cache=False).logits[:, -1, :].float()

@torch.no_grad()
def puntuar(tok, modelo, ids_letra, texto, etiquetas):
    """Etiqueta elegida, su probabilidad entre las 11 letras, masa total de las letras y los 5 tokens más probables."""
    enc = tok([texto], return_tensors='pt', add_special_tokens=False).to('cuda')   # una sola cita: sin relleno
    logits = ultimo_logit(modelo, enc)[0]
    p_vocab = logits.softmax(-1)                                     # probabilidad sobre TODO el vocabulario
    todas = sorted({i for ids in ids_letra for i in ids})
    masa = float(p_vocab[todas].sum())                               # cuánto pesan las 11 letras en lo que el modelo quiere escribir
    p_letras = torch.stack([torch.logsumexp(logits[ids], -1) for ids in ids_letra]).softmax(-1)
    k = int(p_letras.argmax())
    top = p_vocab.topk(5)
    top5 = ' | '.join(f'{tok.convert_ids_to_tokens(int(i))!r} {float(p):.2f}' for p, i in zip(top.values, top.indices))
    return etiquetas[k], round(float(p_letras[k]), 3), round(masa, 3), top5

@torch.no_grad()
def generar(tok, modelo, texto, n=24):
    """Deja que el modelo escriba libremente (sin muestreo) para ver qué responde."""
    enc = tok([texto], return_tensors='pt', add_special_tokens=False).to('cuda')
    out = modelo.generate(**enc, max_new_tokens=n, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True).strip().replace('\n', ' ')

def diagnosticar(nombre):
    tok, modelo, ids_letra = cargar(JUECES[nombre])
    print(f'\n===== {nombre} =====')
    print('Tokens de cada letra:', {l: tok.convert_ids_to_tokens(ids) for l, ids in zip(LETRAS, ids_letra)})
    print('Final del texto que recibe el modelo:', repr(texto_chat(tok, muestra.iloc[0])[-70:]))
    filas = []
    for _, f in muestra.iterrows():
        t_v1 = texto_chat(tok, f)                                    # exactamente lo que vio el juez en la corrida v1
        e1, p1, m1, top1 = puntuar(tok, modelo, ids_letra, t_v1, ETIQUETAS)
        e2, p2, m2, top2 = puntuar(tok, modelo, ids_letra, texto_chat(tok, f, prefijo=PREFIJO), ETIQUETAS)
        e3, p3, m3, _ = puntuar(tok, modelo, ids_letra, texto_chat(tok, f, OPC_INV, PREFIJO), ETQ_INV)
        filas.append({'juez': nombre, 'context_id': f[COL['id']], 'qlora': f['_q'],
                      'v1': e1, 'p_v1': p1, 'masa_v1': m1, 'top_v1': top1,
                      'prefijo': e2, 'p_prefijo': p2, 'masa_prefijo': m2, 'top_prefijo': top2,
                      'invertido': e3, 'p_invertido': p3, 'masa_invertido': m3,
                      'texto_libre': generar(tok, modelo, t_v1)})
    d = pd.DataFrame(filas)
    lotes = [muestra.iloc[i:i + LOTE] for i in range(0, len(muestra), LOTE)]   # mismo método v1, pero en lotes con relleno
    d['v1_en_lote'] = [r['etiqueta'] for l in lotes for r in predecir(tok, modelo, ids_letra, [f for _, f in l.iterrows()])]
    print(f'Masa media de las 11 letras: v1 {d.masa_v1.mean():.2f} | prefijo {d.masa_prefijo.mean():.2f} | invertido {d.masa_invertido.mean():.2f}')
    print(f'Coincide con QLoRA: v1 {(d.v1 == d.qlora).sum()} | prefijo {(d.prefijo == d.qlora).sum()} | de {len(d)}')
    print(f'Misma etiqueta al invertir las opciones: {(d.prefijo == d.invertido).sum()} de {len(d)}')
    print(f'v1 una por una vs en lote, diferencias: {(d.v1 != d.v1_en_lote).sum()}')
    print('Etiquetas v1:', Counter(d.v1).most_common())
    print('Etiquetas con prefijo:', Counter(d.prefijo).most_common())
    del modelo                                                       # libera la GPU para el siguiente juez
    gc.collect()
    torch.cuda.empty_cache()
    return d

diag = pd.concat([diagnosticar(n) for n in JUECES], ignore_index=True)
diag.to_csv(CARPETA_DRIVE / 'diagnostico_jueces.csv', index=False, encoding='utf-8-sig')   # queda en Drive para revisarlo
pd.set_option('display.max_colwidth', 60)
display(diag[['juez', 'qlora', 'v1', 'masa_v1', 'prefijo', 'invertido', 'texto_libre']])

## Paso 5. Corrida completa

**No ejecutes esta sección hasta revisar el diagnóstico del Paso 4b y corregir el método.** Los archivos de la v1 (`juez_llama.jsonl`, `juez_mistral.jsonl`) ya tienen las 18.000 filas, así que esta celda no haría nada con ellos.

Ejecuta cada celda. Si Colab se desconecta, vuelve a montar Drive (Paso 1), ejecuta los Pasos 2 y 3, y repite esta celda: continúa donde quedó.

In [ ]:
llama = correr_juez('llama')
print(len(llama), 'de', len(df))

In [ ]:
mistral = correr_juez('mistral')
print(len(mistral), 'de', len(df))

## Paso 6. Jurado: voto mayoritario y acuerdo entre jueces

Se combinan los tres jueces (QLoRA, Llama y Mistral). La etiqueta final es la que eligen al menos dos de los tres; si los tres difieren, queda `SIN_CONSENSO`. El acuerdo se reporta con el κ de Fleiss (los tres jueces) y el κ de Cohen por pareja.

In [ ]:
def voto_mayoritario(etiquetas):
    """Etiqueta elegida por al menos 2 jueces; si todas difieren, 'SIN_CONSENSO'."""
    etiqueta, votos = Counter(etiquetas).most_common(1)[0]
    return (etiqueta, votos) if votos >= 2 else ('SIN_CONSENSO', 1)

def kappa_cohen(a, b):
    """Acuerdo entre dos jueces corregido por azar."""
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    ca, cb = Counter(a), Counter(b)
    pe = sum(ca[k] * cb[k] for k in set(ca) | set(cb)) / (n * n)
    return (po - pe) / (1 - pe) if pe < 1 else 1.0

def kappa_fleiss(listas):
    """Acuerdo de varios jueces corregido por azar; listas = una lista de etiquetas por juez."""
    m = len(listas)                                                  # número de jueces
    n = len(listas[0])                                               # número de citas
    categorias = sorted({e for l in listas for e in l})
    total_cat = Counter()
    p_i = 0.0
    for i in range(n):
        c = Counter(l[i] for l in listas)
        total_cat.update(c)
        p_i += (sum(v * v for v in c.values()) - m) / (m * (m - 1))
    p_bar = p_i / n
    p_e = sum((total_cat[k] / (n * m)) ** 2 for k in categorias)
    return (p_bar - p_e) / (1 - p_e) if p_e < 1 else 1.0

In [ ]:
def etiqueta_qlora(f):
    """Convierte las columnas del CSV QLoRA a una de las 11 etiquetas."""
    return f[COL['fn']] if f[COL['ev']] == 'Evaluable' else f[COL['ev']]

def particion(cid):
    """Partición de ACL-200 según el año del artículo citante (P13-... = 2013, W99-... = 1999)."""
    a = int(cid[1:3])                                                # dos dígitos del año
    if a == 15:
        return 'test'
    if a == 14:
        return 'val'
    if a <= 13:
        return 'train'                                               # 2000-2013
    return 'anterior_2000'                                           # 1980-1999: aparte hasta confirmar cómo los trata ACL-200

llama, mistral = leer_hechos(CARPETA_DRIVE / 'juez_llama.jsonl'), leer_hechos(CARPETA_DRIVE / 'juez_mistral.jsonl')
filas = []
for _, f in df.iterrows():
    cid = f[COL['id']]
    if cid not in llama or cid not in mistral:
        continue                                                     # solo citas con los tres jueces
    q, l, m = etiqueta_qlora(f), llama[cid]['etiqueta'], mistral[cid]['etiqueta']
    voto, n_votos = voto_mayoritario([q, l, m])
    filas.append({'context_id': cid, 'particion': particion(cid), 'qlora': q, 'llama': l, 'mistral': m,
                  'voto': voto, 'jueces_de_acuerdo': n_votos, 'p_llama': llama[cid]['p'], 'p_mistral': mistral[cid]['p']})
jurado = pd.DataFrame(filas)

print(len(jurado), 'citas con los tres jueces')
print('κ de Fleiss (3 jueces):', round(kappa_fleiss([jurado.qlora.tolist(), jurado.llama.tolist(), jurado.mistral.tolist()]), 3))
for a, b in [('qlora', 'llama'), ('qlora', 'mistral'), ('llama', 'mistral')]:
    print(f'κ de Cohen {a}-{b}:', round(kappa_cohen(jurado[a].tolist(), jurado[b].tolist()), 3))
print('Jueces de acuerdo:', jurado.jueces_de_acuerdo.value_counts().to_dict())
print('Voto por clase y partición:')
print(pd.crosstab(jurado.voto, jurado.particion, margins=True))

jurado.to_csv(CARPETA_DRIVE / 'jurado_18000.csv', index=False, encoding='utf-8-sig')   # resultado final en Drive
print('Guardado en', CARPETA_DRIVE / 'jurado_18000.csv')

## Qué queda en Drive

- `juez_llama.jsonl`, `juez_mistral.jsonl`: una línea por cita con la etiqueta, su probabilidad y la distribución completa.
- `juez_llama_meta.json`, `juez_mistral_meta.json`: modelo, huella del prompt, versiones y GPU usadas (para reproducir).
- `jurado_18000.csv`: las tres etiquetas, el voto y la partición de cada cita.

**Siguiente paso (fuera de este cuaderno):** muestrear el Test Gold desde la partición `test`, estratificado por el voto y excluyendo los contextos ya expuestos. Las citas de `test` **no** se usan para entrenar.